# DeepVRegulome — Batch VCF Scoring Pipeline

This notebook demonstrates how to score all variants in a VCF file against
DeepVRegulome's DNABERT models, filter candidate disruptions, and export results.

**What you'll learn:**
1. Score variants from a VCF file with `dvr.score_vcf()`
2. Multi-GPU parallel inference for large variant sets
3. Filter and rank candidate regulatory disruptions
4. Score against all 458 TF models or all 462 models at once
5. Export results for downstream analysis

**Prerequisites:**
- `pip install deepvregulome[all]`
- A VCF file (`.vcf`, not `.vcf.gz` — decompress first if needed)
- Reference genome FASTA (hg38)

## 1. Setup

In [1]:
import deepvregulome
print(f"deepvregulome version: {deepvregulome.__version__}")

from deepvregulome import DVR
import pandas as pd
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name} ({props.total_memory / 1e9:.1f} GB)")

deepvregulome version: 0.1.8
CUDA available: True
  GPU 0: NVIDIA A40 (47.6 GB)
  GPU 1: NVIDIA A40 (47.6 GB)
  GPU 2: NVIDIA A40 (47.6 GB)
  GPU 3: NVIDIA A40 (47.6 GB)
  GPU 4: NVIDIA A40 (47.6 GB)
  GPU 5: NVIDIA A40 (47.6 GB)
  GPU 6: NVIDIA A40 (47.6 GB)
  GPU 7: NVIDIA A40 (47.6 GB)


In [2]:
# ⚠️ Update these paths
GENOME = "/data/projects/human_reference_genome/hg38/all_chr_ref.fasta"
VCF_FILE ="/home/campus.stonybrook.edu/pdutta/dvr-wrapper/TCGA-LUAD_test.raw_somatic_mutation.vcf"

dvr = DVR(genome=GENOME)

---
## 2. Score VCF Against Selected Models

`dvr.score_vcf()` parses the VCF, runs the coordinate sanity check, extracts
301bp flanking sequences, and scores all variants in batches.

The sanity check auto-detects whether your VCF uses 1-based (standard) or 0-based
coordinates by verifying REF alleles against the reference genome.

In [3]:
# Score all VCF variants against a few TF models
results = dvr.score_vcf(
    VCF_FILE,
    models=["CTCFL", "ATF4"],
    batch_size=1000,
    gpus=[4,5,6,7],      # single GPU; use [0, 1, 2, 3] for multi-GPU
)

print(f"Result shape: {results.shape}")
print(f"Variants scored: {results['pos'].nunique()}")
print(f"Models used: {results['model'].nunique()}")
results.head(10)

Parsing VCF: /home/campus.stonybrook.edu/pdutta/dvr-wrapper/TCGA-LUAD_test.raw_somatic_mutation.vcf
  Found 29101 variants
✓ Sanity check PASSED — 1-based coordinates detected
  Checked 10 variants: 10 confirm 1-based, 0 confirm 0-based, 0 ambiguous, 0 no-match
  REF alleles match the reference genome. Coordinates look correct.
Extracting sequences for 29101 variants...
Scoring 29101 variants × 2 models...
Distributing 2 models across 4 GPUs: [0, 1, 0, 1] each


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 28391.43it/s]


  [GPU 5] ✓ CTCFL (1/1)
  [GPU 7] ✓ ATF4 (1/1)
Result shape: (58202, 10)
Variants scored: 29100
Models used: 2


,chrom,pos,ref,alt,model,type,prob_ref,prob_alt,log_odds_ratio,score_change
0,chr4,57425511,C,A,ATF4,TF,0.000041,0.999962,-29.2643,0.999883
1,chr4,116321140,G,T,ATF4,TF,0.000041,0.999945,-28.7279,0.999849
2,chr4,178264127,C,A,ATF4,TF,0.999962,0.000062,28.6550,-0.999862
3,chr16,51331167,C,G,ATF4,TF,0.000049,0.999950,-28.6011,0.999850
4,chr5,6813857,G,A,ATF4,TF,0.999942,0.000057,28.1775,-0.999827
5,chr5,26860194,C,T,ATF4,TF,0.999986,0.000314,27.7649,-0.999658
6,chr16,36224087,G,C,ATF4,TF,0.000077,0.999936,-27.6021,0.999796
7,chr13,38985481,C,G,ATF4,TF,0.999988,0.000420,27.5308,-0.999556
8,chr12,23930462,T,A,ATF4,TF,0.000078,0.999922,-27.2996,0.999766
9,chr1,72542026,C,A,ATF4,TF,0.999815,0.000044,26.8671,-0.999587


### 3. Filter Candidate Regulatory Disruptions and Gains

A variant is classified based on how it shifts the predicted binding probability across the `0.5` threshold:

* **Disruptive variant (Loss of binding):**
  * `prob_ref > 0.5` and `prob_alt <= 0.5` — crosses the threshold from binding to non-binding.

* **Gain variant (New binding):**
  * `prob_ref < 0.5` and `prob_alt >= 0.5` — crosses the threshold from non-binding to binding.

In [4]:
# ---------------------------------------------------------
# Filter for Disruptive Variants (Loss of binding)
# ---------------------------------------------------------
disrupted = results[(results["prob_ref"] > 0.5) & (results["prob_alt"] <= 0.5)].copy()

print(f"Disrupted variant–TF pairs: {len(disrupted)} / {len(results)}")
print(f"Unique disrupted variants: {disrupted['pos'].nunique()}")

print("\nTop disruptive variant–TF pairs (sorted by largest probability drop):")
# Sorting by the largest drop in probability to see the most severe disruptions first
disrupted["prob_diff"] = disrupted["prob_ref"] - disrupted["prob_alt"]
disrupted_sorted = disrupted.sort_values("prob_diff", ascending=False)

cols = ["chrom", "pos", "ref", "alt", "model", "prob_ref", "prob_alt"]
print(disrupted_sorted[cols].head(20))

Disrupted variant–TF pairs: 407 / 58202
Unique disrupted variants: 403

Top disruptive variant–TF pairs (sorted by largest probability drop):
    chrom        pos ref alt  model  prob_ref  prob_alt
2    chr4  178264127   C   A   ATF4  0.999962  0.000062
4    chr5    6813857   G   A   ATF4  0.999942  0.000057
9    chr1   72542026   C   A   ATF4  0.999815  0.000044
11   chr5   39172566   C   A  CTCFL  0.999820  0.000065
19  chr19   29011868   A   T   ATF4  0.999863  0.000126
16   chr7  143075532   C   T  CTCFL  0.999761  0.000067
5    chr5   26860194   C   T   ATF4  0.999986  0.000314
17   chr6   86372763   G   T  CTCFL  0.999728  0.000060
21   chr5  113814384   G   T  CTCFL  0.999685  0.000063
14   chr4   34162370   G   A   ATF4  0.999663  0.000044
24  chr11   37308699   G   A  CTCFL  0.999687  0.000069
28   chr7   64332317   G   T  CTCFL  0.999880  0.000265
27   chr1   99086707  TC  AA   ATF4  0.999670  0.000082
7   chr13   38985481   C   G   ATF4  0.999988  0.000420
20  chr21    91251

In [5]:
# ---------------------------------------------------------
# Filter for Gain Variants (New binding)
# ---------------------------------------------------------
gain = results[(results["prob_ref"] < 0.5) & (results["prob_alt"] >= 0.5)].copy()

print(f"\nGain variant–TF pairs: {len(gain)} / {len(results)}")
print(f"Unique gain variants: {gain['pos'].nunique()}")

print("\nTop gain variant–TF pairs (sorted by largest probability jump):")
# Sorting by the largest jump in probability
gain["prob_diff"] = gain["prob_alt"] - gain["prob_ref"]
gain_sorted = gain.sort_values("prob_diff", ascending=False)

print(gain_sorted[cols].head(20))


Gain variant–TF pairs: 358 / 58202
Unique gain variants: 358

Top gain variant–TF pairs (sorted by largest probability jump):
    chrom        pos ref alt  model  prob_ref  prob_alt
0    chr4   57425511   C   A   ATF4  0.000041  0.999962
1    chr4  116321140   G   T   ATF4  0.000041  0.999945
3   chr16   51331167   C   G   ATF4  0.000049  0.999950
6   chr16   36224087   G   C   ATF4  0.000077  0.999936
8   chr12   23930462   T   A   ATF4  0.000078  0.999922
13   chr5  166113289   C   A   ATF4  0.000146  0.999900
15  chr18   69491398   A   G   ATF4  0.000202  0.999925
22   chr1  242257388   C   A   ATF4  0.000185  0.999893
12   chr2  155372377   G   T   ATF4  0.000053  0.999754
25   chr2  203690385   A   G  CTCFL  0.000075  0.999708
29  chr12   84554201   C   G  CTCFL  0.000142  0.999767
34   chr1  124632428   A   C  CTCFL  0.000279  0.999841
35  chr12   43604735   A   G   ATF4  0.000264  0.999815
32   chr2  211955005   C   A   ATF4  0.000090  0.999569
30  chr13   54307100   G   T   AT

In [6]:
# Rank by severity (strongest disruptions first)
disrupted_sorted = disrupted.sort_values("log_odds_ratio", ascending=False)

print("Top 20 most disruptive variant–TF pairs:")
cols = ["chrom", "pos", "ref", "alt", "model", "prob_ref", "prob_alt", "log_odds_ratio", "score_change"]
disrupted_sorted[cols].head(20)

Top 20 most disruptive variant–TF pairs:


,chrom,pos,ref,alt,model,prob_ref,prob_alt,log_odds_ratio,score_change
2,chr4,178264127,C,A,ATF4,0.999962,0.000062,28.6550,-0.999862
4,chr5,6813857,G,A,ATF4,0.999942,0.000057,28.1775,-0.999827
5,chr5,26860194,C,T,ATF4,0.999986,0.000314,27.7649,-0.999658
7,chr13,38985481,C,G,ATF4,0.999988,0.000420,27.5308,-0.999556
9,chr1,72542026,C,A,ATF4,0.999815,0.000044,26.8671,-0.999587
11,chr5,39172566,C,A,CTCFL,0.999820,0.000065,26.3561,-0.999575
14,chr4,34162370,G,A,ATF4,0.999663,0.000044,26.0007,-0.999281
16,chr7,143075532,C,T,CTCFL,0.999761,0.000067,25.8949,-0.999456
17,chr6,86372763,G,T,CTCFL,0.999728,0.000060,25.8663,-0.999395
18,chr2,171380079,C,G,ATF4,0.999346,0.000026,25.8071,-0.998667


In [7]:
# Count how many TFs are disrupted per variant
variant_summary = (
    disrupted
    .groupby(["chrom", "pos", "ref", "alt"])
    .agg(
        n_disrupted_tfs=("model", "nunique"),
        max_log_odds=("log_odds_ratio", lambda x: x.abs().max()),
        disrupted_tfs=("model", lambda x: ", ".join(sorted(x.unique()))),
    )
    .sort_values("n_disrupted_tfs", ascending=False)
    .reset_index()
)

print(f"Variants disrupting at least one TF: {len(variant_summary)}")
variant_summary.head(10)

Variants disrupting at least one TF: 403


,chrom,pos,ref,alt,n_disrupted_tfs,max_log_odds,disrupted_tfs
0,chr6,46935925,G,T,2,5.8862,"ATF4, CTCFL"
1,chr5,5579073,G,T,2,19.5534,"ATF4, CTCFL"
2,chrX,82932034,C,T,2,15.8629,"ATF4, CTCFL"
3,chr4,4336748,G,T,2,18.1703,"ATF4, CTCFL"
4,chr5,7194121,TG,CA,1,2.5701,ATF4
5,chr5,6813857,G,A,1,28.1775,ATF4
6,chr5,5484346,G,C,1,4.2898,ATF4
7,chr5,4991526,G,T,1,13.3688,CTCFL
8,chr5,4036609,G,A,1,12.9470,ATF4
9,chr4,189389480,G,T,1,24.7082,ATF4


---
## 4. Score Against All TF Models

For comprehensive regulatory profiling, score against all 458 TFs.
This takes longer but gives a complete picture of which binding sites each variant disrupts.

> **Timing estimate:** ~29K variants × 458 models ≈ 4–6 hours on a single A100.
> Using 4 GPUs reduces this to ~1–1.5 hours.

In [ ]:
# Score against ALL TF models (uncomment to run)
# results_all_tf = dvr.score_vcf(
#     VCF_FILE,
#     model_type="TF",         # all 458 TF models
#     batch_size=100,
#     gpus=[0, 1, 2, 3],       # multi-GPU
# )

# Or all 462 models (TF + histone):
# results_all = dvr.score_vcf(
#     VCF_FILE,
#     model_type=None,          # None = all models
#     batch_size=100,
#     gpus=[0, 1, 2, 3],
# )

---
## 5. Working with VCF Data Tips

### Compressed VCF files
DeepVRegulome currently accepts uncompressed `.vcf` files. If your file is compressed:

```bash
# Decompress
bgzip -d your_file.vcf.gz
# Or use gunzip
gunzip your_file.vcf.gz
```

### Filtering VCF before scoring
For large VCFs, pre-filter to save time:

```bash
# Keep only PASS variants
bcftools view -f PASS input.vcf > filtered.vcf

# Keep only variants in a region
bcftools view -r chr9:60000000-70000000 input.vcf > region.vcf

# Keep only SNVs (no indels)
bcftools view -v snps input.vcf > snvs_only.vcf
```

---
## 6. Export Results

In [ ]:
# Save full results
results.to_csv("deepvregulome_results.tsv", sep="\t", index=False)
print("Saved: deepvregulome_results.tsv")

# Save disrupted candidates only
disrupted_sorted.to_csv("deepvregulome_candidates.tsv", sep="\t", index=False)
print(f"Saved: deepvregulome_candidates.tsv ({len(disrupted_sorted)} rows)")

# Save variant summary
variant_summary.to_csv("deepvregulome_variant_summary.tsv", sep="\t", index=False)
print(f"Saved: deepvregulome_variant_summary.tsv ({len(variant_summary)} variants)")

---
## Summary

**Workflow:**
1. `dvr.score_vcf()` — parses VCF, auto-detects coordinates, scores in batches
2. Filter by `|log_odds_ratio| > 2.0` for candidate disruptions
3. Rank by severity and count disrupted TFs per variant
4. Export for downstream clinical/functional analysis

**Performance tips:**
- Use `batch_size=100–320` depending on GPU memory
- Use multiple GPUs: `gpus=[0, 1, 2, 3]` distributes models across GPUs
- Pre-filter VCF with `bcftools` for large datasets

**Next:** See `03_attention_motifs.ipynb` for attention-based interpretability and motif analysis.